# 01 파일·환경 점검
v3 산출물과 대표 슬라이드 751장(SVS·임베딩)의 존재를 확인하고, 주 분석 venv의 패키지와 GPU를 점검한다.
- 커널: `C:\Users\Admin\EXAONEPath\exaonepath-venv`
- 출력: `C:\Users\Admin\EXAONEPath_data\v4\01_inventory\`

In [1]:
# 1. 경로
from pathlib import Path
import sys, shutil, subprocess, json
from importlib.metadata import version, PackageNotFoundError
import pandas as pd, h5py

V3   = Path(r"C:\Users\Admin\EXAONEPath_data\v3")
SH   = Path(r"C:\Users\Admin\EXAONEPath_data\_shared")
PY10 = Path(r"C:\Users\Admin\EXAONEPath\exaonepath-venv\Scripts\python.exe")
OUT  = Path(r"C:\Users\Admin\EXAONEPath_data\v4\01_inventory")
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
# 2. 이후 분석에 쓸 핵심 파일
per_c = [r"labels\{c}_labels_final.csv", r"labels\{c}_patient_table.csv", r"labels\{c}_meth_labels.csv",
         r"omics\{c}_rna_tpm.parquet", r"omics\{c}_meth450_beta.h5", r"features_pooled\{c}_pooled.h5",
         r"features_cache\{c}_index.csv", r"features_cache\{c}_tiles.f16", r"features_cache\{c}_xy.i32",
         r"splits\{c}_folds.csv"]
key  = [V3 / p.format(c=c) for c in ("luad", "lusc") for p in per_c]
key += [V3 / p for p in (r"manifest\slides.csv", r"manifest\patients.csv",
                         r"splits\lung_site5fold.json", r"qc\06_qc_flags.csv")]
key += [SH / f for f in ("tcga_cdr.xlsx", "absolute_purity.tsv", "pancan_clinical.tsv", "sample_quality.tsv",
                         "thorsson_mmc2.xlsx", "estimate_genesets.gmt", "HM450.hg38.manifest.tsv.gz")]

kf = pd.DataFrame({"path": [str(p) for p in key], "exists": [p.exists() for p in key],
                   "MB": [round(p.stat().st_size / 2**20, 2) if p.exists() else None for p in key]})
kf.to_csv(OUT / "key_files.csv", index=False, encoding="utf-8-sig")
print(f"핵심 파일 {kf.exists.sum()}/{len(kf)} 존재")
kf[~kf.exists]

핵심 파일 31/31 존재


,path,exists,MB


In [3]:
# 3. 폴더별 파일 수·용량 (v3 1단계, luad·lusc 2단계, _shared)
dirs = [d for d in V3.iterdir() if d.is_dir()] + [d for c in ("luad", "lusc") for d in (V3 / c).iterdir() if d.is_dir()] + [SH]
rows = []
for d in dirs:
    fs = [f for f in d.rglob("*") if f.is_file()]
    rows.append((str(d), len(fs), round(sum(f.stat().st_size for f in fs) / 2**30, 2)))
inv = pd.DataFrame(rows, columns=["dir", "files", "GB"]).sort_values("dir")
inv.to_csv(OUT / "dirs.csv", index=False, encoding="utf-8-sig")
inv

,dir,files,GB
29,C:\Users\Admin\EXAONEPath_data\_shared,10,0.06
0,C:\Users\Admin\EXAONEPath_data\v3\cell_features,9,0.00
1,C:\Users\Admin\EXAONEPath_data\v3\features_cache,6,12.38
2,C:\Users\Admin\EXAONEPath_data\v3\features_pooled,2,0.01
3,C:\Users\Admin\EXAONEPath_data\v3\figures,47,0.59
4,C:\Users\Admin\EXAONEPath_data\v3\labels,15,0.00
5,C:\Users\Admin\EXAONEPath_data\v3\legacy,1,0.00
6,C:\Users\Admin\EXAONEPath_data\v3\logs,15,0.00
7,C:\Users\Admin\EXAONEPath_data\v3\luad,4835,307.80
19,C:\Users\Admin\EXAONEPath_data\v3\luad\feature...,403,5.96


In [4]:
# 4. 다운로드한 대표 슬라이드별 SVS 크기 일치와 임베딩 타일 수
sl  = pd.read_csv(V3 / r"manifest\slides.csv")
rep = pd.read_csv(V3 / r"manifest\patients.csv")["rep_slide_id"].dropna()
s = sl[sl.slide_id.isin(rep)].copy()
s["c"] = s.cancer.str.lower()
s["svs"] = [V3 / c / "svs" / n for c, n in zip(s.c, s.file_name)]
s = s[s.svs.map(Path.exists)].copy()  # 다운로드한 751장만

def n_tiles(p):  # -1: 파일 없음·손상, 0: 조직 타일 없음
    try:
        with h5py.File(p, "r") as f: return f["features"].shape[0]
    except Exception: return -1

s["svs_ok"] = [p.stat().st_size == z for p, z in zip(s.svs, s.file_size)]
s["n_tiles"] = [n_tiles(V3 / c / "features_10" / f"{i}.h5") for c, i in zip(s.c, s.slide_id)]
s[["slide_id", "patient_id", "cancer", "tss", "svs_ok", "n_tiles"]].to_csv(
    OUT / "slides_status.csv", index=False, encoding="utf-8-sig")

summ = s.groupby("cancer").agg(slides=("slide_id", "size"), svs_ok=("svs_ok", "sum"),
                               embedded=("n_tiles", lambda x: int((x > 0).sum())))
print(summ)
s.loc[~s.svs_ok | (s.n_tiles <= 0), ["slide_id", "cancer", "svs_ok", "n_tiles"]]

        slides  svs_ok  embedded
cancer                          
LUAD       404     404       403
LUSC       347     347       347


,slide_id,cancer,svs_ok,n_tiles
369,TCGA-44-7661-01Z-00-DX1,LUAD,True,-1


In [5]:
# 5. venv 점검: 커널 경로, 패키지 버전, GPU, 디스크
print("커널:", sys.executable, "| venv 일치:", Path(sys.executable).resolve() == PY10.resolve())
print("Python", sys.version.split()[0])

PKGS = ["torch", "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "lifelines", "h5py", "pyarrow",
        "openslide-python", "opencv-python", "scikit-image", "gseapy", "pyreadr", "matplotlib", "seaborn",
        "requests", "openpyxl", "upsetplot"]
def ver(p):
    try: return version(p)
    except PackageNotFoundError: return None
pk = pd.DataFrame({"package": PKGS, "version": [ver(p) for p in PKGS]})
print("미설치:", pk[pk.version.isna()].package.tolist())

import torch
gpu = {"cuda": torch.cuda.is_available(), "torch_cuda": torch.version.cuda,
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if torch.cuda.is_available() else None}
free_gb = round(shutil.disk_usage("C:\\").free / 2**30, 1)
print(gpu, f"| C: 여유 {free_gb} GB")

(OUT / "env10_freeze.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout, encoding="utf-8")
pk

커널: C:\Users\Admin\EXAONEPath\exaonepath-venv\Scripts\python.exe | venv 일치: True
Python 3.14.6
미설치: ['upsetplot']
{'cuda': True, 'torch_cuda': '12.6', 'device': 'NVIDIA GeForce RTX 3060', 'vram_gb': 12.0} | C: 여유 121.9 GB


,package,version
0,torch,2.14.0+cu126
1,numpy,2.5.2
2,pandas,3.0.5
3,scipy,1.18.1
4,scikit-learn,1.9.1
5,statsmodels,0.15.0
6,lifelines,0.30.0
7,h5py,3.16.0
8,pyarrow,26.0.0
9,openslide-python,1.4.6


In [6]:
# 6. 요약 저장
summary = {"key_files_missing": kf.loc[~kf.exists, "path"].tolist(),
           "per_cancer": summ.reset_index().to_dict("records"),
           "python": sys.version.split()[0], "missing_packages": pk[pk.version.isna()].package.tolist(),
           "gpu": gpu, "disk_free_gb": free_gb}
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=int), encoding="utf-8")
summary

{'key_files_missing': [],
 'per_cancer': [{'cancer': 'LUAD',
   'slides': 404,
   'svs_ok': 404,
   'embedded': 403},
  {'cancer': 'LUSC', 'slides': 347, 'svs_ok': 347, 'embedded': 347}],
 'python': '3.14.6',
 'missing_packages': ['upsetplot'],
 'gpu': {'cuda': True,
  'torch_cuda': '12.6',
  'device': 'NVIDIA GeForce RTX 3060',
  'vram_gb': 12.0},
 'disk_free_gb': 121.9}